# 05. Ground Truth 평가셋 구축

**목표**: TF-IDF vs Sentence-BERT 비교 실험의 평가 기준이 될 쿼리-정답 쌍 구축

| 항목 | 내용 |
|------|------|
| 평가셋 크기 | 쿼리 50개 (생애주기별 고르게 분포) |
| 구축 방식 | 반자동 — 후보 자동 생성 후 수동 검토 |
| 평가 지표 | Top-1 정확도, Top-3 정확도, MRR |
| 저장 파일 | `data/splits/ground_truth.csv` |

> **왜 필요한가**: Ground Truth 없이는 "BERT가 더 좋다"는 주장의 근거가 없음.  
> 이 파일이 `06_matching.ipynb`과 `07_evaluation.ipynb`의 기준이 됨.

## 0. 환경 설정

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from pathlib import Path

from utils.config import DATA_PROCESSED, PROJECT_ROOT

PREPROCESSED_PATH  = DATA_PROCESSED / 'corpus_preprocessed.csv'
SPLITS_DIR         = PROJECT_ROOT / 'data/splits'
GROUND_TRUTH_PATH  = SPLITS_DIR / 'ground_truth.csv'

SPLITS_DIR.mkdir(parents=True, exist_ok=True)
print("환경 설정 완료")

## 1. 데이터 로드

In [ ]:
df = pd.read_csv(PREPROCESSED_PATH)
print(f"전체 레코드: {len(df):,}개")

# 평가셋은 train split만 사용 (validation은 매칭 대상 DB로 사용)
df_train = df[df['split'] == 'train'].reset_index(drop=True)
df_val   = df[df['split'] == 'validation'].reset_index(drop=True)
print(f"Train: {len(df_train):,} / Validation: {len(df_val):,}")

## 2. 후보 쿼리 자동 생성

생애주기별 균등 샘플링 → 수동 검토용 후보 목록 생성

In [ ]:
QUERIES_PER_LIFECYCLE = 17  # 자견 17 + 성견 17 + 노령견 16 = 50개

candidates = []
for lc, n in [('자견', 17), ('성견', 17), ('노령견', 16)]:
    sample = df_val[df_val['lifeCycle'] == lc].sample(
        n=min(n, len(df_val[df_val['lifeCycle'] == lc])),
        random_state=42
    )
    candidates.append(sample)

df_candidates = pd.concat(candidates, ignore_index=True)
print(f"후보 쿼리 생성: {len(df_candidates)}개")
print(df_candidates['lifeCycle'].value_counts())

## 3. Ground Truth 구조 생성

In [ ]:
# ground_truth DataFrame 구조
# query_id    : 고유 ID
# query       : 실제 질문 텍스트
# lifeCycle   : 생애주기
# department  : 진료과
# disease     : 질병
# correct_idx : 정답 레코드의 train index (수동 검토 후 기입)
# correct_input: 정답 레코드의 input 텍스트 (확인용)

df_gt = df_candidates[['input', 'lifeCycle', 'department', 'disease']].copy()
df_gt = df_gt.rename(columns={'input': 'query'})
df_gt.insert(0, 'query_id', [f'Q{i+1:03d}' for i in range(len(df_gt))])

# 정답 컬럼 — 수동 검토 전 -1로 초기화
df_gt['correct_idx']   = -1
df_gt['correct_input'] = ''
df_gt['verified']      = False  # 수동 검토 완료 여부

print(f"Ground Truth 구조:")
print(df_gt.columns.tolist())
df_gt.head(3)

## 4. 자동 정답 후보 매칭 (수동 검토 보조)

같은 disease + lifeCycle을 가진 train 레코드를 정답 후보로 자동 연결.  
이후 수동으로 적절성 검토 필요.

In [ ]:
def find_answer_candidate(row: pd.Series, db: pd.DataFrame) -> pd.Series:
    """disease + lifeCycle 조건으로 train DB에서 정답 후보 1개 반환."""
    matches = db[
        (db['disease'] == row['disease']) &
        (db['lifeCycle'] == row['lifeCycle'])
    ]
    if len(matches) > 0:
        best = matches.sample(1, random_state=42).iloc[0]
        return pd.Series({
            'correct_idx':   int(best.name),
            'correct_input': best['input'][:100] + '...'
        })
    return pd.Series({'correct_idx': -1, 'correct_input': '정답 없음'})

auto_matched = df_gt.apply(find_answer_candidate, axis=1, db=df_train)
df_gt['correct_idx']   = auto_matched['correct_idx']
df_gt['correct_input'] = auto_matched['correct_input']

matched = (df_gt['correct_idx'] != -1).sum()
print(f"자동 매칭 성공: {matched}/{len(df_gt)}개 ({matched/len(df_gt)*100:.0f}%)")

## 5. 수동 검토 가이드

아래 셀을 실행하면 검토가 필요한 항목을 출력.  
각 쿼리와 후보 정답을 확인 후 `verified = True`로 수정.

In [ ]:
# 검토 대상 출력 (처음 5개)
for _, row in df_gt.head(5).iterrows():
    print(f"{'='*60}")
    print(f"[{row['query_id']}] {row['lifeCycle']} / {row['department']} / {row['disease']}")
    print(f"\n[질문]\n{row['query'][:200]}")
    print(f"\n[정답 후보]\n{row['correct_input']}")
    print(f"정답 idx: {row['correct_idx']}")

In [ ]:
# 수동 검토 완료 후 verified = True 일괄 설정
# (실제 검토 후 개별 수정 또는 아래처럼 일괄 처리)
df_gt.loc[df_gt['correct_idx'] != -1, 'verified'] = True

verified_count = df_gt['verified'].sum()
print(f"검토 완료: {verified_count}/{len(df_gt)}개")

## 6. 저장

In [ ]:
df_gt.to_csv(GROUND_TRUTH_PATH, index=False, encoding='utf-8-sig')
print(f"저장 완료: {GROUND_TRUTH_PATH}")
print(f"\n생애주기별 분포:")
print(df_gt['lifeCycle'].value_counts().to_string())
df_gt.head()

## 7. 평가 지표 미리보기

`06_matching.ipynb`에서 이 ground_truth를 기준으로 아래 지표를 계산:

| 지표 | 설명 |
|------|------|
| **Top-1 정확도** | 1순위 추천이 정답인 비율 |
| **Top-3 정확도** | 3순위 안에 정답이 포함된 비율 |
| **MRR** | 정답이 몇 번째에 나왔는지 역수의 평균 (1위=1.0, 2위=0.5, 3위=0.33) |

```python
# MRR 계산 예시
def mean_reciprocal_rank(results: list[list], ground_truth: list) -> float:
    rr_list = []
    for pred, true in zip(results, ground_truth):
        if true in pred:
            rank = pred.index(true) + 1
            rr_list.append(1 / rank)
        else:
            rr_list.append(0)
    return sum(rr_list) / len(rr_list)
```

> **다음 단계**: `06_matching.ipynb` — TF-IDF vs Sentence-BERT 매칭 실험